# XLCoST results — one language per session

**Runtime → GPU (L4)**, set `LANGUAGE` in cell 2, then **Runtime → Run all**.
Every step is resumable; re-run after a disconnect and it continues.

## Running several languages in parallel

Open this notebook in **separate browser tabs**, set a different `LANGUAGE`
in each, and run them concurrently. This is safe by construction: all
artifact names are language-scoped, and the save cell copies only the
current language's files, so sessions never overwrite each other.

- Extractors exist for **Python, Java, Javascript, PHP**. C/C++/C# have none.
- The **renaming experiment (C1–C5) is Python-only** for now; other
  languages run the baseline pass (probe + model-free baselines) and the
  sweep cell skips renaming automatically.
- Each session consumes compute units separately (~1.5/hr each) and needs
  its own GPU. If GPUs are scarce, a **CPU runtime is enough** for any
  language whose activation stores are already on Drive — probing and
  baselines never touch the GPU.

In [ ]:
# 1 — setup: clone/pull main, deps, token, restore prior work from Drive
# Reproduction/audit runs: set PIN_COMMIT to a full SHA to run vetted code
# instead of the moving branch. Use a READ-only HF token in Colab secrets.
PIN_COMMIT = ""
import os, pathlib
REPO = "/content/mech-interp"
if not pathlib.Path(REPO).exists():
    !git clone -q -b main https://github.com/nolanlwin/mech-interp.git {REPO}
%cd {REPO}
!git fetch -q origin
_old = !git rev-parse HEAD
if PIN_COMMIT:
    !git checkout -q {PIN_COMMIT}
else:
    !git checkout -q main 2>/dev/null || git checkout -q -b main origin/main
    !git pull -q
_new = !git rev-parse HEAD
if _old[0] != _new[0]:
    print("=" * 70)
    print("CODE CHANGED since this runtime last ran — review before trusting")
    print("the run with your HF token / Drive. New commits:")
    !git log --oneline {_old[0]}..{_new[0]}
    print("=" * 70)
!git log --oneline -1
!pip install -q transformers==5.8.0 tree_sitter "tree-sitter-java>=0.23.5" "tree-sitter-go>=0.25.0" \
  "tree-sitter-javascript>=0.25.0" "tree-sitter-php>=0.24.1" "tree-sitter-ruby>=0.23.1" \
  scikit-learn scipy huggingface_hub
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass
from google.colab import drive
drive.mount("/content/drive")
DEST = "/content/drive/MyDrive/mech-interp/xlcost"
!mkdir -p outputs/activations_xlcost outputs/probe_results outputs/xlcost_occ data/xlcost outputs/xlcost_occ_renamed data/xlcost_renamed
!cp -rn {DEST}/stores/* outputs/activations_xlcost/ 2>/dev/null || true
!cp -n {DEST}/probe_results/* outputs/probe_results/ 2>/dev/null || true
!cp -n {DEST}/xlcost_occ/* outputs/xlcost_occ/ 2>/dev/null || true
!cp -n {DEST}/data_xlcost/* data/xlcost/ 2>/dev/null || true
!cp -rn {DEST}/xlcost_occ_renamed/* outputs/xlcost_occ_renamed/ 2>/dev/null || true
!cp -rn {DEST}/data_xlcost_renamed/* data/xlcost_renamed/ 2>/dev/null || true
print("setup complete")

In [ ]:
# 2 — CONFIG: one language per session (see the parallel-runs note above)
LANGUAGE = "Python"        # Python | Java | Javascript | PHP
SPLIT = "train"
MODELS = [
    "Qwen/Qwen2.5-Coder-1.5B",
    "Qwen/Qwen2.5-1.5B",      # base-vs-Coder contrast at identical scale
    "bigcode/starcoder2-7b",  # different family, tokenizer, pretraining corpus
]

slug = LANGUAGE.lower().replace("++", "pp").replace("#", "sharp")
mslug = lambda m: m.split("/")[-1].lower().replace(".", "").replace("-", "")
RENAMING = LANGUAGE == "Python"   # renamer v1 scope
print(f"{LANGUAGE}/{SPLIT} | {len(MODELS)} model(s) | renaming: {RENAMING}")

In [ ]:
# 3 — the sweep: baseline pass (+ renaming for Python) per model. GPU.
#     Checkpoints to Drive after each model so a disconnect cannot lose one.
for m in MODELS:
    print(f"\n############ {LANGUAGE} — {m} ############")
    !bash scripts/run_language.sh {LANGUAGE} {m} {SPLIT}
    if RENAMING:
        !bash scripts/run_renaming.sh {LANGUAGE} {m} {SPLIT}
    print(f"checkpoint: saving {LANGUAGE}/{m} to Drive")
    !mkdir -p {DEST}/probe_results {DEST}/stores
    !cp -r outputs/probe_results/{slug}_{SPLIT}_* {DEST}/probe_results/ 2>/dev/null || true
    !cp -r outputs/activations_xlcost/{slug}_{SPLIT}_* {DEST}/stores/ 2>/dev/null || true

In [ ]:
# 4 — results table for this language (reads whatever exists)
import glob, json, re
probes = sorted(glob.glob(f"outputs/probe_results/{slug}_{SPLIT}_*_problem.json"))
models = sorted({m.group(1) for f in probes
                 if (m := re.match(rf".*{slug}_{SPLIT}_(?!C\d)(\w+)_problem\.json$", f))})
hdr = f"{'model':<20}{'C0 F1':>8}{'select.':>9}{'best base':>11}"
if RENAMING:
    hdr += "".join(f"{c:>9}" for c in ["dC1", "dC2", "dC3", "dC4", "dC5"])
print(f"=== {LANGUAGE}/{SPLIT} ===\n" + hdr)
for m in models:
    c0 = json.load(open(f"outputs/probe_results/{slug}_{SPLIT}_{m}_problem.json"))
    f1 = c0["aggregate"]["test_macro_f1_mean"]
    sel = c0.get("selectivity_macro_f1", float("nan"))
    try:
        best = json.load(open(f"outputs/probe_results/{slug}_{SPLIT}_{m}_baselines_capped.json"))["strongest_baseline_macro_f1"]
    except FileNotFoundError:
        best = float("nan")
    row = f"{m:<20}{f1:>8.4f}{sel:>9.4f}{best:>11.4f}"
    if RENAMING:
        for c in ["C1", "C2", "C3", "C4", "C5"]:
            try:
                d = json.load(open(f"outputs/probe_results/{slug}_{SPLIT}_{c}_{m}_delta_vs_C0.json"))
                row += f"{d['delta']:>+9.4f}"
            except FileNotFoundError:
                row += f"{'—':>9}"
    print(row)
    print(f"{'':<20}producing commit: {c0.get('git_commit', '(unstamped)')[:12]}")
print("\ndeltas = condition minus C0, paired on shared test occurrences (CIs in *_delta_vs_C0.json)")

In [ ]:
# 5 — save this language's artifacts to Drive (safe to run concurrently
#     with other languages' sessions: every path is language-scoped)
!mkdir -p {DEST}/probe_results {DEST}/stores {DEST}/xlcost_occ {DEST}/data_xlcost {DEST}/xlcost_occ_renamed {DEST}/data_xlcost_renamed
!cp -r outputs/probe_results/{slug}_{SPLIT}_* {DEST}/probe_results/ 2>/dev/null || true
!cp -r outputs/xlcost_occ/{slug}_{SPLIT}* {DEST}/xlcost_occ/ 2>/dev/null || true
!cp -r data/xlcost/{slug}_{SPLIT}* {DEST}/data_xlcost/ 2>/dev/null || true
!cp -r outputs/activations_xlcost/{slug}_{SPLIT}_* {DEST}/stores/ 2>/dev/null || true
for c in ["C1", "C2", "C3", "C4", "C5"]:
    !mkdir -p {DEST}/xlcost_occ_renamed/{c} {DEST}/data_xlcost_renamed/{c}
    !cp -r outputs/xlcost_occ_renamed/{c}/{slug}_{SPLIT}* {DEST}/xlcost_occ_renamed/{c}/ 2>/dev/null || true
    !cp -r data/xlcost_renamed/{c}/{slug}_{SPLIT}* {DEST}/data_xlcost_renamed/{c}/ 2>/dev/null || true
!ls {DEST}/probe_results/ | grep "^{slug}_" | tail -6
print(f"saved {LANGUAGE}/{SPLIT}")